In [43]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
import numpy as np
import pandas as pd
import math

Get the gene indices from Sc_genome_annotations

In [30]:
inputfile = open('/Users/rl884/Downloads/Sc_genome_annotations.txt', 'r')
GeneNames = []
for line in inputfile:
    if line.split('\t')[5] == '':
        GeneNames.append(line.split('\t')[4])
    else:
        GeneNames.append(line.split('\t')[5])
inputfile.close()
GeneNames = GeneNames[1:]
#print(GeneNames)

Get the TFs in the NDR/NFR for each Tandem gene

In [47]:
inputfile = open('/Users/rl884/Downloads/Tandem_Genes_and_Stuff_inbetween.txt', 'r')

TF_Gene_Matrix = {}
for line in inputfile:
    if line.split('\t')[5] == '':
        genename = line.split('\t')[4]
    else:
        genename = line.split('\t')[5]
    temp_TFs = []
    for each in line.split('\t')[8:-1]:
        temp_TFs.append(each.split('_')[0])
    temp_TFs = set(temp_TFs)
    temp_vector = np.zeros(shape=np.array(GeneNames).shape)
    for each in temp_TFs:
        if each.upper() == 'NUC':
            continue
        else:
            temp_vector[GeneNames.index(each.upper())] = 1
    if genename in TF_Gene_Matrix:
        raise Exception('Existed gene!')
    else:
        TF_Gene_Matrix[genename] = temp_vector
inputfile.close()

[[ 985]
 [2643]
 [4509]]


Get the TFs in the NDR/NFR for each pair of H-H gene

In [48]:
inputfile = open('/Users/rl884/Downloads/Divergent_Genes_and_Stuff_inbetween.txt', 'r')
for line in inputfile:
    temp_vector_left = np.zeros(shape=np.array(GeneNames).shape)
    temp_vector_right = np.zeros(shape=np.array(GeneNames).shape)
    left_ = int(line.split('\t')[3])
    right_ = int(line.split('\t')[9])
    insulator_ = []
    
    if abs(right_-left_) <= 300:
        # the NDR in between is short, consider the genes as co-regulated.
        pass
    else:
        for each in line.split('\t')[10:-1]:
            # judge the rap1, reb1, and abf1: insulator in the middle or repressor close to a gene.
            if (each.split('_')[0] in ['Rap1', 'Reb1', 'Abf1']) and abs(int(each.split('_')[-1])-0.5*(left_+right_)) <= 0.15*abs(right_-left_):
                insulator_.append(int(each.split('_')[-1]))
            elif each.split('_')[0] in ['Rap1', 'Reb1', 'Abf1']:
                if abs(int(each.split('_')[-1])-left_) < abs(int(each.split('_')[-1])-right_):
                    temp_vector_left[GeneNames.index(each.split('_')[0].upper())] = 1
                else:
                    temp_vector_right[GeneNames.index(each.split('_')[0].upper())] = 1
            else:
                pass
            
    for each in line.split('\t')[10:-1]:
        if each.split('_')[0] not in ['Rap1', 'Reb1', 'Abf1', 'nuc']:
            if len(insulator_) == 0:
                temp_vector_left[GeneNames.index(each.split('_')[0].upper())] = 1
                temp_vector_right[GeneNames.index(each.split('_')[0].upper())] = 1
            else:
                if abs(int(each.split('_')[-1])-left_) < abs(min(insulator_)-left_):
                    temp_vector_left[GeneNames.index(each.split('_')[0].upper())] = 1
                elif abs(int(each.split('_')[-1])-right_) < abs(max(insulator_)-right_):
                    temp_vector_right[GeneNames.index(each.split('_')[0].upper())] = 1
                else:
                    print('buried in insulator.')
        else:
            pass
    # for the left gene:
    genename = line.split('\t')[0]
    if genename in TF_Gene_Matrix:
        TF_Gene_Matrix[genename] = TF_Gene_Matrix[genename] + temp_vector_left
    else:
        TF_Gene_Matrix[genename] = temp_vector_left
    #for the right gene:
    genename = line.split('\t')[5]
    if genename in TF_Gene_Matrix:
        TF_Gene_Matrix[genename] = TF_Gene_Matrix[genename] + temp_vector_right
    else:
        TF_Gene_Matrix[genename] = temp_vector_right
    
inputfile.close()

[[ 985]
 [2643]
 [4509]]


In [41]:
inputfile = open('/Users/rl884/Downloads/PICIN/ssTFs_common_names.txt', 'r')
ssTF_names = []
for line in inputfile:
    ssTF_names.append(line.split()[0])
inputfile.close()

In [49]:
OutMatrix = []
GeneNames.index(ssTF_names[0])
for each_row in ssTF_names:
    OutMatrix.append([])
    for each_column in ssTF_names:
        OutMatrix[-1].append(TF_Gene_Matrix[each_row][GeneNames.index(each_column)])
OutMatrix = pd.DataFrame(np.array(OutMatrix).T, index=ssTF_names, columns=ssTF_names)
OutMatrix.to_excel('/Users/rl884/Downloads/2021-Rossi_Nature-master/04_ChExMix_Peaks/ssTFs_ChIP.xlsx')

In [50]:
OutMatrix

,ABF1,AFT2,AZF1,BAS1,CAD1,CBF1,CHA4,CIN5,CRZ1,CUP9,...,TBS1,TEA1,UME6,URC2,WAR1,YAP1,YAP7,YRR1,ZAP1,AFT1
ABF1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AFT2,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
AZF1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
BAS1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
CAD1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
YAP1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
YAP7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
YRR1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
ZAP1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [66]:
YEP_input = pd.read_excel('/Users/rl884/Downloads/2021-Rossi_Nature-master/04_ChExMix_Peaks/YEP_Network.xlsx', header=0, index_col=0)
YEP_input

82

In [68]:
for each in np.argwhere((OutMatrix - YEP_input) == -1):
    print(ssTF_names[each[0]], '->', ssTF_names[each[1]])


HAP2 -> GCN4
HAP3 -> GCN4
HAP5 -> GCN4
NRG1 -> PDR1
NRG2 -> PDR1
RAP1 -> GCN4
RAP1 -> MATALPHA2
RPN4 -> PDR1
RPN4 -> REB1
RPN4 -> YAP1
SFP1 -> GCN4
SKO1 -> PDR3
SNT2 -> FHL1
STB5 -> GCN4
TEA1 -> GCN4
YRR1 -> TBS1
